# Experiment Durchführung


In [1]:
# Autoreload: übernimmt Änderungen an experiment/*.py ohne Kernel-Neustart
# (verhindert TypeError/NameError durch veraltete, gecachte Module).
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
from dotenv import load_dotenv
from experiment.llm_connector import LLMConnector
from experiment.data_loader import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ROOT

PosixPath('/Users/pascalknoll/Documents/Knoll_Bachelorarbeit/Python_Bachelorarbeit')

## Artikel laden

In [2]:
loader = DataLoader(
    raw_path       = "../data/raw",
    processed_path = "../data/processed",
)

print("Verfügbare Artikel (raw):      ", loader.list_available_articles("raw"))
print("Verfügbare Artikel (processed):", loader.list_available_articles("processed"))


Verfügbare Artikel (raw):       ['03_Original.html', '02_Original.html', '01_Original.html']
Verfügbare Artikel (processed): ['01_AuthoritativeTone.html', '01_QuotationAddition.html', '02_ConclusionFirst.html', '01_JSON-LD.html', '01_FluencyOptimization.html', '02_AuthoritativeTone.html', '02_QuotationAddition.html', '03_QuotationAddition.html', '03_AuthoritativeTone.html', '03_ConclusionFirst.html', '03_StatisticsAddition.html', '03_Original.html', '02_LogicalStructure.html', '02_FluencyOptimization.html', '02_Original.html', '01_StatisticsAddition.html', '01_Original.html', '02_JSON-LD.html', '01_LogicalStructure.html', '02_StatisticsAddition.html', '03_LogicalStructure.html', '03_FluencyOptimization.html', '03_JSON-LD.html', '01_ConclusionFirst.html']


In [3]:
# Alle Artikel bereinigen und in data/processed speichern
#processing_stats = loader.process_all()

#print(f"\n{len(processing_stats)} Dateien verarbeitet.")

[OK] raw/03_Original.html -> 1973 Token (geschätzt)
[OK] raw/02_Original.html -> 4164 Token (geschätzt)
[OK] raw/01_Original.html -> 2008 Token (geschätzt)

3 Dateien verarbeitet.


In [4]:
#import pandas as pd
#from IPython.display import display, HTML, Markdown
# Token-Statistiken als DataFrame
#stats_df = (
#    pd.DataFrame(processing_stats)
#    .T
#    .reset_index()
#    .rename(columns={"index": "datei"})
#    .sort_values(["quelle", "datei"])
#)

#display(stats_df)

,datei,zeichen,token_geschaetzt,quelle
2,01_Original.html,8034,2008,raw
1,02_Original.html,16657,4164,raw
0,03_Original.html,7892,1973,raw


## Einzelne LLM-Anfrage testen

Die folgenden Zellen benoetigen passende API-Keys. Lege dafuer im Projektordner eine `.env` nach dem Muster aus `.env.example` an.

In [5]:
#
#questions = [
#    "Worum geht es in diesem Artikel?",
#    "Welche zentralen Fakten werden genannt?",
#]
#
#system_prompt = "Du bist ein praeziser Analyst. Antworte kurz und faktenbasiert."
#user_prompt = f"Artikel:\n{content}\n\nFrage: {questions[0]}"


In [3]:
# Smoke-Test OpenAI: prüft Modell-ID (gpt-5.4-mini-2026-03-17), Seed-Parameter und Token-Protokollierung.
# Bewusst manuell ausführen – verursacht einen einzelnen, minimalen API-Call.
chatgpt = LLMConnector(provider="chatgpt")
result = chatgpt.send_query("You are a test assistant.", "Reply with the single word OK.", seed=42)
print(result["text"], "|", result["prompt_tokens"], "Input-Tokens,", result["completion_tokens"], "Output-Tokens")

OK | 23 Input-Tokens, 4 Output-Tokens


In [5]:
# Smoke-Test Gemini: prüft Modell-ID (gemini-3.5-flash), Seed-Parameter und Token-Protokollierung.
# Bewusst manuell ausführen – verursacht einen einzelnen, minimalen API-Call.
gemini = LLMConnector(provider="gemini")
result = gemini.send_query("You are a test assistant.", "Reply with the single word OK.", seed=42)
print(result["text"], "|", result["prompt_tokens"], "Input-Tokens,", result["completion_tokens"], "Output-Tokens")

OK | 15 Input-Tokens, 94 Output-Tokens


## Experiment ausführen – kontrolliert je Durchlauf

Versuchsumfang gemäß Abschnitt 4.3.1 der Thesis: 3 Artikel × 8 Varianten × 2 Modelle × 5 Seeds = **240 API-Aufrufe**, aufgeteilt in **5 einzeln startbare Durchläufe** à 48 Aufrufe (ein Durchlauf = ein Seed). Zellen von oben nach unten ausführen (Zelle 1 und „Artikel laden" müssen im aktuellen Kernel gelaufen sein).

Kostenkontrolle:

- **`runner.estimate()`** – kostenlose Schätzung der Input-Tokens; sobald ein Durchlauf vollständig ist, Hochrechnung der restlichen aus den Ist-Werten (inkl. Reasoning-Tokens).
- **`runner.run_full_experiment(dry_run=True)`** – kompletter Probelauf ohne API-Aufrufe (Dateien mit Suffix `_DRYRUN`).
- **`runner.run_replication(N)`** – führt genau Durchlauf N aus (Dateien `experiment_results_runN_seedS.csv`). Vollständige Durchläufe werden übersprungen, unvollständige nur aufgefüllt (Absturz-/Fehler-Nachholung) – doppeltes Bezahlen ist ausgeschlossen. Neuerheben nur mit `force=True`.
- **`runner.status()`** – Fortschritt, Token-Verbrauch und (bei gepflegter `PRICES`-Liste) Kosten je Durchlauf.

Fehlgeschlagene Aufrufe werden bis zu dreimal wiederholt und landen andernfalls im separaten Error-Log (`experiment_errors_runN_seedS.csv`) – nie im Ergebnisdatensatz; das nächste `run_replication(N)` holt genau diese Lücken nach.

In [4]:
from experiment.experiment_runner import ExperimentRunner

# System- und User-Prompt gemäß Kapitel 4.3.2 der Thesis (Prompts pr:system-prompt
# und pr:user-prompt) – konstant über alle 240 Aufrufe; {article} wird zur Laufzeit
# durch die jeweilige Artikelvariante aus data/processed ersetzt.
SYSTEM_PROMPT = (
    "You are a precise and objective information extraction assistant. "
    "Your task is to analyze press articles and answer questions strictly based on the provided article content. "
    "Do not use any external knowledge, assumptions, or information that is not explicitly stated in the article. "
    "If a piece of information is not contained in the article, state that it is not mentioned rather than inferring or fabricating it."
)

USER_PROMPT = """Below is a press article provided as HTML content. Please read it carefully and answer the following questions based solely on the article.

[ARTICLE START]
{article}
[ARTICLE END]

Based exclusively on the article above, please provide:
1. A concise summary of the article's key messages (list each key message as a separate bullet point).
2. The most important factual details mentioned (e.g., numbers, dates, product names, technical specifications).
3. A brief statement on which company or institution is the author or source of this press release."""

# Kanonischer Seed-Satz: n = 5 Replikationen, Durchlauf N nutzt SEEDS[N-1]
# identisch für jede Variante und beide Modelle (Abschnitt 4.3.1 der Thesis).
SEEDS = [42, 43, 44, 45, 46]

# Optional: Preise in EUR je 1 Mio. Tokens aus den aktuellen Preislisten der
# Anbieter eintragen – dann rechnen status()/estimate() Tokens in Kosten um.
# Die Schlüssel müssen exakt den protokollierten Modellversionen entsprechen.
PRICES = {
    "gpt-5.4-mini-2026-03-17": {"input": None, "output": None},
    "gemini-3.5-flash":        {"input": None, "output": None},
}

gemini = LLMConnector(provider="gemini")    # gemini-3.5-flash (stabiler Alias, keine datierten Snapshots verfügbar)
chatgpt = LLMConnector(provider="chatgpt")  # gpt-5.4-mini-2026-03-17 (datierter Snapshot)

runner = ExperimentRunner(
    loader=loader,
    connector_list=[gemini, chatgpt],
    prompt_config={
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": USER_PROMPT,
    },
    # Thematische Kategorie je Artikel (Spalte Article_Type im CSV-Schema)
    article_types={
        "01": "Product Launch",
        "02": "Production",
        "03": "Sustainability",
    },
    seeds=SEEDS,
    prices=PRICES,
)

# Kostenlose Vorab-Schätzung; nach dem ersten vollständigen Echtlauf
# rechnet estimate() die restlichen Durchläufe aus den Ist-Werten hoch.
runner.estimate()

Aufrufe je Durchlauf: 48  |  Durchläufe: 5 (5 vollständig, 0 ausstehend)
Geschätzte Input-Tokens je Durchlauf (Zeichen/4-Heuristik): ~147,550
Ist-Werte aus 5 vollständigen Durchläufen: ~2,708 Input- + ~1,406 Output-Tokens je Aufruf
Hochrechnung je weiterem Durchlauf: ~197,468 Tokens; für 0 ausstehende: ~0 Tokens


{'calls_per_run': 48,
 'estimated_input_tokens_per_run': 147550,
 'runs_completed': 5,
 'runs_remaining': 0,
 'actual_avg_prompt_tokens': 2707.625,
 'actual_avg_completion_tokens': 1406.3,
 'projected_tokens_per_run': 197468.40000000002}

In [5]:
# DRY RUN (kostenlos): kompletter Probelauf aller 5 Durchläufe ohne API-Aufrufe.
# Erzeugt separate Dateien mit Suffix _DRYRUN und dient als Pipeline-Check,
# bevor echte Kosten entstehen. Erwartung: 240 Aufrufe, 0 Fehler.
runner.run_full_experiment(dry_run=True)

--- Experiment gestartet am 28.07.2026 14:03 (DRY RUN, keine API-Aufrufe): 5 Durchläufe x 48 Aufrufe ---
--- Durchlauf 1 (Seed 42): 48 von 48 Aufrufen ausstehend (DRY RUN, keine API-Aufrufe) ---
[1/48] 01_AuthoritativeTone.html | gemini-3.5-flash
[2/48] 01_AuthoritativeTone.html | gpt-5.4-mini-2026-03-17
[3/48] 01_ConclusionFirst.html | gemini-3.5-flash
[4/48] 01_ConclusionFirst.html | gpt-5.4-mini-2026-03-17
[5/48] 01_FluencyOptimization.html | gemini-3.5-flash
[6/48] 01_FluencyOptimization.html | gpt-5.4-mini-2026-03-17
[7/48] 01_JSON-LD.html | gemini-3.5-flash
[8/48] 01_JSON-LD.html | gpt-5.4-mini-2026-03-17
[9/48] 01_LogicalStructure.html | gemini-3.5-flash
[10/48] 01_LogicalStructure.html | gpt-5.4-mini-2026-03-17
[11/48] 01_Original.html | gemini-3.5-flash
[12/48] 01_Original.html | gpt-5.4-mini-2026-03-17
[13/48] 01_QuotationAddition.html | gemini-3.5-flash
[14/48] 01_QuotationAddition.html | gpt-5.4-mini-2026-03-17
[15/48] 01_StatisticsAddition.html | gemini-3.5-flash
[16/48] 0

In [12]:
# ECHTLAUF: Durchlauf RUN_NR ausführen (48 API-Aufrufe, kostenpflichtig!).
# Workflow: Durchlauf 1 starten -> runner.status() prüfen -> erst dann RUN_NR
# erhöhen (1..5). Vollständige Durchläufe werden automatisch übersprungen,
# unvollständige nur aufgefüllt – doppeltes Bezahlen ist ausgeschlossen.
# Bewusstes Neuerheben eines Durchlaufs nur mit force=True.
RUN_NR = 5

runner.run_replication(RUN_NR, dry_run=False)

--- Durchlauf 5 (Seed 46): 48 von 48 Aufrufen ausstehend (ECHTLAUF, API-Aufrufe verursachen Kosten) ---
[1/48] 01_AuthoritativeTone.html | gemini-3.5-flash
[WARNUNG] gemini-3.5-flash: Versuch 1/3 fehlgeschlagen (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}). Nächster Versuch in 2s.
[WARNUNG] gemini-3.5-flash: Versuch 2/3 fehlgeschlagen (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}). Nächster Versuch in 4s.
[2/48] 01_AuthoritativeTone.html | gpt-5.4-mini-2026-03-17
[3/48] 01_ConclusionFirst.html | gemini-3.5-flash
[WARNUNG] gemini-3.5-flash: Versuch 1/3 fehlgeschlagen (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are u

In [1]:
# Fortschritts- und Kostenübersicht über alle Durchläufe (liest die CSV-Dateien;
# Token-Zahlen sind Ist-Werte aus den API-Antworten, inkl. Reasoning-Tokens).
runner.status()

NameError: name 'runner' is not defined